In [42]:
import pandas as pd
from keras.callbacks import EarlyStopping
from keras.layers import Dense, Dropout
from keras.models import Sequential
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [43]:
data = pd.read_csv("C:/Sarang/College/lp3/ml/Churn_Modelling.csv")

print("Dataset shape:", data.shape)
print(data.head())

Dataset shape: (10000, 14)
   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0       2       0.00              1          1               1   
1       1   83807.86              1          0               1   
2       8  159660.80              3          1               0   
3       1       0.00              2          0               0   
4       2  125510.82              1          1               1   

   EstimatedSalary  Exited  
0        101348.88       1  
1        112542.58       0  
2        113931.57       1  
3         93826

In [44]:
X = data.drop(["RowNumber", "CustomerId", "Surname", "Exited"], axis=1)

y = data["Exited"]


X = pd.get_dummies(
    X,
    columns=["Geography", "Gender"],
    drop_first=True
)

print("\nFeatures:")
print(X.columns)


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


Features:
Index(['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Geography_Germany',
       'Geography_Spain', 'Gender_Male'],
      dtype='str')


In [45]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


print("\nTraining data:", X_train.shape)
print("Testing data:", X_test.shape)


Training data: (8000, 11)
Testing data: (2000, 11)


In [46]:
model = Sequential()

model.add(
    Dense(
        64,
        activation="relu",
        input_shape=(X_train.shape[1],)
    )
)

model.add(Dropout(0.30))

model.add(
    Dense(
        32,
        activation="relu"
    )
)

model.add(Dropout(0.20))

model.add(
    Dense(
        1,
        activation="sigmoid"
    )
)


model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


model.summary()

C:\Users\sarve\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_15 (Dense)                     │ (None, 64)                  │             768 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_10 (Dropout)                 │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_16 (Dense)                     │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_11 (Dropout)                 │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_17 (Dense)                     │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,881 (11.25 KB)

 Trainable params: 2,881 (11.25 KB)

 Non-trainable params: 0 (0.00 B)

In [47]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)


history = model.fit(
    X_train,
    y_train,
    validation_split=0.20,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7769 - loss: 0.4907 - val_accuracy: 0.8163 - val_loss: 0.4301
Epoch 2/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8100 - loss: 0.4370 - val_accuracy: 0.8363 - val_loss: 0.4119
Epoch 3/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8211 - loss: 0.4199 - val_accuracy: 0.8388 - val_loss: 0.3981
Epoch 4/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8264 - loss: 0.4065 - val_accuracy: 0.8550 - val_loss: 0.3748
Epoch 5/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8353 - loss: 0.3955 - val_accuracy: 0.8587 - val_loss: 0.3601
Epoch 6/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8420 - loss: 0.3844 - val_accuracy: 0.8606 - val_loss: 0.3506
Epoch 7/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8436 - loss: 0.3797 - val_accuracy: 0.8662 - val_loss: 0.3441
Epoch 8/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8442 - loss: 0.3748 - val_accu

In [48]:
y_pred_prob = model.predict(X_test)

y_pred = (y_pred_prob >= 0.5).astype(int).ravel()


accuracy = accuracy_score(
    y_test,
    y_pred
)

print("\nAccuracy:", accuracy)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

Accuracy: 0.861


In [49]:
cm = confusion_matrix(
    y_test,
    y_pred
)

print("\nConfusion Matrix:")
print(cm)


Confusion Matrix:
[[1539   54]
 [ 224  183]]


In [ ]:
# ============================================================
# BANK CUSTOMER CHURN PREDICTION USING A NEURAL NETWORK
# VIVA NOTES / EXPLANATIONS
# ============================================================
#
# IMPORTANT FILE-PRESERVATION NOTE
# - The original 8 code cells above this cell are preserved exactly.
# - This cell contains comments only. It does not execute or change the model.
# - The notebook reads a CSV file; it does not ask the user to type records
#   interactively.
# - The CSV path in the original loading cell is specific to one Windows
#   computer. To run on another computer, the path in that original cell must
#   point to the location of Churn_Modelling.csv. The original cell has not
#   been edited here.
#
# ============================================================
# 1. PROJECT OBJECTIVE
# ============================================================
#
# This project predicts whether a bank customer will leave the bank.
#
# - This task is called customer churn prediction.
# - The target column is "Exited".
# - In the commonly used Bank Churn dataset:
#       Exited = 1 means the customer left the bank.
#       Exited = 0 means the customer stayed.
# - The model is a feed-forward Artificial Neural Network (ANN).
# - Since the target has two classes (0 and 1), this is a binary
#   classification problem.
#
# A simple explanation to the external examiner:
# "This project uses customer information such as age, credit score,
# geography, gender and account-related details to predict whether a
# customer is likely to leave the bank. I preprocess the data, split it
# into training and testing sets, scale the features, train a neural
# network, and evaluate its predictions using accuracy and a confusion
# matrix."
#
# ============================================================
# 2. CELL-BY-CELL / LINE-BY-LINE EXPLANATION
# ============================================================
#
# -------------------------
# CELL 0: IMPORT LIBRARIES
# -------------------------
#
# import pandas as pd
# - Imports the Pandas library and gives it the short name pd.
# - Pandas is used to load and manipulate tabular data such as CSV files.
#
# from keras.callbacks import EarlyStopping
# - Imports EarlyStopping, a training callback.
# - It monitors a selected training metric and stops training when that
#   metric stops improving for a specified number of epochs.
#
# from keras.layers import Dense, Dropout
# - Dense is a fully connected neural-network layer.
# - Dropout randomly disables a fraction of layer outputs during training
#   to help reduce overfitting.
#
# from keras.models import Sequential
# - Imports the Sequential model.
# - Sequential is used when neural-network layers are arranged one after
#   another in a linear stack.
#
# from sklearn.metrics import accuracy_score, confusion_matrix
# - accuracy_score calculates the fraction of correct predictions.
# - confusion_matrix counts correct and incorrect predictions for each
#   class.
#
# from sklearn.model_selection import train_test_split
# - Imports the function used to divide the dataset into training and
#   testing subsets.
#
# from sklearn.preprocessing import StandardScaler
# - Imports StandardScaler, which standardizes numerical features using
#   statistics calculated from the training data.
#
# -------------------------
# CELL 1: LOAD THE DATASET
# -------------------------
#
# data = pd.read_csv("C:/Sarang/College/lp3/ml/Churn_Modelling.csv")
# - Reads the CSV file and stores its contents in a Pandas DataFrame
#   named data.
# - The path shown is a local Windows path. It must exist on the machine
#   running the notebook, or it must be changed to the dataset's actual
#   location when setting up another machine.
#
# print("Dataset shape:", data.shape)
# - Displays the dataset dimensions as (number of rows, number of columns).
# - Each row usually represents one customer.
# - Each column represents a customer attribute or the target.
#
# print(data.head())
# - Displays the first five rows by default.
# - This helps inspect the column names and example records.
#
# -------------------------
# CELL 2: SEPARATE FEATURES AND TARGET, ENCODE, SPLIT
# -------------------------
#
# X = data.drop(["RowNumber", "CustomerId", "Surname", "Exited"], axis=1)
# - Creates X, the input-feature DataFrame.
# - Removes RowNumber, CustomerId and Surname because they are identifiers
#   or names rather than useful numerical model inputs.
# - Removes Exited from X because Exited is the answer the model must learn
#   to predict.
# - axis=1 means the listed columns are removed from the columns axis.
# - The original data DataFrame is not deleted by this statement.
#
# y = data["Exited"]
# - Creates y, the target/output series.
# - The neural network learns to predict this column.
#
# X = pd.get_dummies(
#     X,
#     columns=["Geography", "Gender"],
#     drop_first=True
# )
# - Converts the categorical Geography and Gender columns into numeric
#   indicator columns. This process is called one-hot encoding.
# - Machine-learning estimators generally require numeric input features.
# - drop_first=True drops one category from each encoded feature to avoid
#   redundant indicator columns.
# - The category dropped becomes the reference/baseline category.
# - The exact dummy-column names depend on the categories present in the
#   dataset.
#
# print("\nFeatures:")
# print(X.columns)
# - Prints the final feature names after encoding.
# - This is useful to check which columns will be supplied to the model.
#
# X_train, X_test, y_train, y_test = train_test_split(
#     X,
#     y,
#     test_size=0.20,
#     random_state=42,
#     stratify=y
# )
# - Splits the features and target into training and testing sets.
# - test_size=0.20 reserves 20% of the records for testing and uses 80%
#   for training.
# - random_state=42 makes the random split reproducible.
# - stratify=y keeps approximately the same class proportions in the
#   training and testing sets as in the complete target.
# - X_train and y_train are used to train the model.
# - X_test and y_test are held back for final evaluation.
#
# -------------------------
# CELL 3: FEATURE SCALING
# -------------------------
#
# scaler = StandardScaler()
# - Creates a StandardScaler object.
# - Standardization transforms a feature using:
#       z = (x - mean) / standard_deviation
# - After standardization, a feature is generally centered around zero
#   and has unit standard deviation, subject to the data distribution.
#
# X_train = scaler.fit_transform(X_train)
# - fit calculates the mean and standard deviation from the training data.
# - transform applies those statistics to the training data.
# - The result is a scaled numeric array.
# - Fitting only on training data helps prevent data leakage.
#
# X_test = scaler.transform(X_test)
# - Applies the already learned training-set scaling parameters to test data.
# - Do NOT fit a new scaler on the test data: the test set must remain
#   independent of the training process.
#
# print("\nTraining data:", X_train.shape)
# print("Testing data:", X_test.shape)
# - Prints the dimensions of the scaled training and testing feature arrays.
# - The number of feature columns should be the same in both arrays.
#
# -------------------------
# CELL 4: BUILD AND COMPILE THE NEURAL NETWORK
# -------------------------
#
# model = Sequential()
# - Creates an empty Sequential neural-network model.
# - Layers are added in the order in which data passes through them.
#
# model.add(Dense(64, activation="relu",
#                 input_shape=(X_train.shape[1],)))
# - Adds the first fully connected (Dense) hidden layer.
# - 64 is the number of neurons in this layer.
# - ReLU is the activation function.
# - ReLU is defined as max(0, x); it outputs zero for negative inputs
#   and the input itself for positive inputs.
# - input_shape tells Keras how many input features each record has.
# - X_train.shape[1] is the number of feature columns.
#
# model.add(Dropout(0.30))
# - Adds a dropout layer with a rate of 0.30.
# - During training, approximately 30% of the inputs to this layer are
#   randomly set to zero.
# - This regularization technique can reduce overfitting.
# - Dropout is inactive during normal inference/prediction.
#
# model.add(Dense(32, activation="relu"))
# - Adds a second hidden Dense layer containing 32 neurons.
# - ReLU adds non-linearity so the network can learn more complex patterns.
#
# model.add(Dropout(0.20))
# - Adds another dropout layer with a rate of 0.20.
# - It randomly drops approximately 20% of the layer inputs during training.
#
# model.add(Dense(1, activation="sigmoid"))
# - Adds the output layer with one neuron.
# - One output neuron is suitable for binary classification.
# - Sigmoid converts the output to a value between 0 and 1.
# - This value is interpreted as the model's estimated probability for
#   the positive class (class 1).
#
# model.compile(
#     optimizer="adam",
#     loss="binary_crossentropy",
#     metrics=["accuracy"]
# )
# - Configures how the neural network is trained.
# - Adam is the optimizer; it updates model weights to reduce the loss.
# - binary_crossentropy is the loss function for binary classification.
# - accuracy is tracked as an additional metric during training.
#
# model.summary()
# - Prints the network architecture, output shapes and parameter counts.
# - Trainable parameters are the weights and biases learned during training.
#
# -------------------------
# CELL 5: EARLY STOPPING AND MODEL TRAINING
# -------------------------
#
# early_stop = EarlyStopping(
#     monitor="val_loss",
#     patience=10,
#     restore_best_weights=True
# )
# - Creates an EarlyStopping callback.
# - monitor="val_loss" means it watches validation loss after each epoch.
# - patience=10 means training may continue for up to 10 epochs without
#   improvement before stopping.
# - restore_best_weights=True restores the weights from the epoch with
#   the best monitored validation loss.
#
# history = model.fit(
#     X_train,
#     y_train,
#     validation_split=0.20,
#     epochs=100,
#     batch_size=32,
#     callbacks=[early_stop],
#     verbose=1
# )
# - Starts neural-network training.
# - X_train and y_train are the training features and labels.
# - validation_split=0.20 sets aside 20% of the supplied training data
#   for validation during training.
# - epochs=100 is the maximum number of passes through the training data;
#   EarlyStopping may stop training earlier.
# - batch_size=32 means the model processes 32 training records per
#   gradient update, except possibly the final smaller batch.
# - callbacks=[early_stop] activates the EarlyStopping callback.
# - verbose=1 displays a progress bar and training metrics.
# - history stores the training history returned by Keras.
#
# NOTE ABOUT THE SPLIT:
# - 20% of the full dataset is initially held out as the test set.
# - Of the remaining 80%, validation_split=0.20 reserves 20% for validation.
# - Approximately 64% of the full dataset is therefore used for fitting,
#   16% for validation, and 20% for testing.
# - The test set is not used by model.fit().
#
# -------------------------
# CELL 6: PREDICTION AND ACCURACY
# -------------------------
#
# y_pred_prob = model.predict(X_test)
# - Uses the trained model to predict probabilities for the test records.
# - With a sigmoid output, each prediction is between 0 and 1.
#
# y_pred = (y_pred_prob >= 0.5).astype(int).ravel()
# - Applies a threshold of 0.5 to convert probabilities into class labels.
# - Probability >= 0.5 becomes class 1.
# - Probability < 0.5 becomes class 0.
# - astype(int) converts Boolean results into integer labels 0 and 1.
# - ravel() flattens the prediction array into a one-dimensional array.
#
# accuracy = accuracy_score(y_test, y_pred)
# - Compares predicted labels with the actual test labels.
# - Calculates the proportion of test records classified correctly.
#
# print("\nAccuracy:", accuracy)
# - Displays the calculated test accuracy.
# - The notebook's actual accuracy depends on the CSV data, preprocessing,
#   software versions, and training run. No fixed result is guaranteed.
#
# -------------------------
# CELL 7: CONFUSION MATRIX
# -------------------------
#
# cm = confusion_matrix(y_test, y_pred)
# - Builds a confusion matrix by comparing actual and predicted labels.
# - For the usual binary labels [0, 1], the matrix is arranged as:
#                    Predicted 0    Predicted 1
#       Actual 0         TN             FP
#       Actual 1         FN             TP
# - TN = True Negative: actual 0, predicted 0.
# - FP = False Positive: actual 0, predicted 1.
# - FN = False Negative: actual 1, predicted 0.
# - TP = True Positive: actual 1, predicted 1.
#
# print("\nConfusion Matrix:")
# print(cm)
# - Displays the confusion matrix.
# - In churn prediction, a false negative means a customer who actually
#   leaves (Exited=1) was predicted to stay (Exited=0).
#
# ============================================================
# 3. COMPLETE PROGRAM LOGIC FOR EXTERNAL VIVA
# ============================================================
#
# You can explain the program in this order:
#
# 1. "First, I import Pandas, Keras and Scikit-learn libraries."
# 2. "I load the bank customer churn CSV file into a DataFrame."
# 3. "I separate the independent variables/features X from the dependent
#    variable/target y. The target is Exited."
# 4. "I remove identifier columns and encode Geography and Gender using
#    one-hot encoding."
# 5. "I split the data into 80% training and 20% testing data, while
#    stratifying by the target class."
# 6. "I standardize the features. The scaler is fitted only on training
#    data and then applied to the test data."
# 7. "I build a Sequential ANN with two hidden Dense layers containing
#    64 and 32 neurons. Both use ReLU activation. Dropout layers are
#    added for regularization."
# 8. "The output layer has one neuron with sigmoid activation because
#    this is binary classification."
# 9. "I compile the network using the Adam optimizer, binary
#    cross-entropy loss and accuracy metric."
# 10. "I train the network with a validation split and EarlyStopping
#     to help prevent unnecessary training and overfitting."
# 11. "I predict probabilities for the test set and use 0.5 as the
#     classification threshold."
# 12. "Finally, I calculate test accuracy and display the confusion matrix."
#
# ONE-MINUTE VIVA ANSWER:
# "This project predicts bank customer churn using an Artificial Neural
# Network. I load the customer dataset, separate the Exited target,
# remove identifier columns, encode categorical features and split the
# data into training and testing sets. I standardize the features using
# training-set statistics. Then I train a Sequential neural network with
# 64 and 32 ReLU neurons, dropout regularization and a one-neuron sigmoid
# output. I use Adam and binary cross-entropy, with EarlyStopping based
# on validation loss. Finally, I convert predicted probabilities into
# class labels and evaluate the model using accuracy and a confusion
# matrix."
#
# ============================================================
# 4. IMPORTANT CONCEPTS, DEFINITIONS AND FORMULAS
# ============================================================
#
# A. CUSTOMER CHURN
# - Churn means a customer stops using a company's service or leaves it.
# - Churn prediction identifies customers who may leave.
# - A bank can use such predictions to plan customer-retention efforts.
#
# B. BINARY CLASSIFICATION
# - A supervised machine-learning task with two possible target classes.
# - Here, the classes are 0 (not exited) and 1 (exited).
#
# C. ARTIFICIAL NEURAL NETWORK (ANN)
# - A machine-learning model inspired by interconnected neurons.
# - It contains input, hidden and output layers.
# - It learns patterns by adjusting weights and biases during training.
#
# D. DENSE / FULLY CONNECTED LAYER
# - Every neuron in a Dense layer receives input from every output in
#   the previous layer.
# - The layer calculates a weighted sum plus bias, then applies an
#   activation function.
#
# E. ACTIVATION FUNCTION
# - A function applied to a neuron's weighted input.
# - It introduces non-linearity, allowing a neural network to learn
#   non-linear relationships.
#
# F. ReLU
# - Formula: ReLU(x) = max(0, x)
# - Commonly used in hidden layers.
# - It outputs zero for negative values and x for positive values.
#
# G. SIGMOID
# - Formula: sigmoid(x) = 1 / (1 + e^(-x))
# - Its output lies between 0 and 1.
# - It is commonly used for a single-output binary classifier.
#
# H. DROPOUT
# - A regularization method used during training.
# - It randomly sets some layer outputs to zero.
# - It helps reduce dependence on particular neurons and can reduce
#   overfitting.
# - A dropout rate of 0.30 means 30% are randomly dropped during training.
#
# I. OVERFITTING
# - A model learns the training data too closely, including noise.
# - It performs well on training data but generalizes poorly to unseen data.
# - Dropout, validation and EarlyStopping can help reduce this problem.
#
# J. UNDERFITTING
# - A model is too simple or insufficiently trained to learn the patterns.
# - It performs poorly on both training and unseen data.
#
# K. STANDARDIZATION
# - Formula: z = (x - mean) / standard_deviation
# - It puts numerical features on a comparable scale.
# - StandardScaler learns mean and standard deviation from training data.
#
# L. DATA LEAKAGE
# - Data leakage occurs when information from outside the training data
#   improperly influences model training.
# - Fitting StandardScaler on the entire dataset before splitting would
#   leak information from the test data.
# - This notebook avoids that particular issue by fitting on X_train only.
#
# M. ONE-HOT ENCODING
# - Converts categories into separate binary indicator columns.
# - Example: a categorical column with categories A, B and C can be
#   represented by indicator columns.
# - drop_first=True removes one category as the reference category.
#
# N. TRAINING, VALIDATION AND TESTING
# - Training set: used to learn model weights.
# - Validation set: used to monitor training and make model-selection
#   decisions such as EarlyStopping.
# - Test set: held out to estimate performance on unseen records.
#
# O. EPOCH
# - One complete pass through the training data used by the training loop.
#
# P. BATCH SIZE
# - Number of training records processed in one batch before a weight update.
# - Here, batch_size=32.
#
# Q. OPTIMIZER: ADAM
# - Adam is an adaptive optimization algorithm.
# - It adjusts model weights using gradient information and adaptive
#   estimates of first and second moments.
# - Its purpose is to minimize the training loss.
#
# R. LOSS FUNCTION: BINARY CROSS-ENTROPY
# - Used for binary classification with a probability output.
# - For one example:
#       Loss = -[y log(p) + (1-y) log(1-p)]
#   where y is the true label and p is the predicted probability for class 1.
# - The training process attempts to minimize this loss.
#
# S. EARLY STOPPING
# - Monitors a metric such as validation loss.
# - Stops training when the metric does not improve for the chosen patience.
# - restore_best_weights=True returns to the best monitored epoch's weights.
#
# T. ACCURACY
# - Formula:
#       Accuracy = (TP + TN) / (TP + TN + FP + FN)
# - It measures the fraction of predictions that are correct.
# - Accuracy alone can be misleading when classes are imbalanced.
#
# U. CONFUSION MATRIX
# - Summarizes classification results by comparing true and predicted labels.
# - For binary classification it contains TN, FP, FN and TP.
#
# V. PRECISION, RECALL AND F1-SCORE
# - Precision = TP / (TP + FP)
# - Recall = TP / (TP + FN)
# - F1 = 2 * Precision * Recall / (Precision + Recall)
# - These are useful additional metrics, especially when the classes
#   are imbalanced.
# - The current notebook calculates accuracy and a confusion matrix;
#   it does not calculate these three metrics.
#
# W. CLASS THRESHOLD
# - The notebook uses 0.5 to convert predicted probabilities to labels.
# - The threshold can be changed depending on the business cost of false
#   positives and false negatives.
# - A threshold should be selected using validation data, not by tuning
#   against the final test set.
#
# ============================================================
# 5. EXPECTED EXTERNAL VIVA QUESTIONS WITH ANSWERS
# ============================================================
#
# Q1. What is the aim of this project?
# Ans: To predict whether a bank customer will leave the bank using
# customer information and a neural-network classification model.
#
# Q2. What is customer churn?
# Ans: Customer churn means a customer stops using a company's service
# or ends the relationship with that company.
#
# Q3. What type of machine-learning problem is this?
# Ans: Supervised binary classification.
#
# Q4. Why is it supervised learning?
# Ans: Because the dataset contains a known target label, Exited, for
# each training record.
#
# Q5. What does Exited represent?
# Ans: It is the target column indicating whether the customer exited.
# In the commonly used dataset, 1 means exited and 0 means not exited.
#
# Q6. What are features and labels?
# Ans: Features are the input variables used for prediction. The label
# is the output we want the model to predict. Here, X contains features
# and y contains Exited.
#
# Q7. Why are RowNumber, CustomerId and Surname removed?
# Ans: They are identifiers or names, not meaningful numeric measurements
# for this model. Keeping them could add irrelevant information.
#
# Q8. Why is Exited removed from X?
# Ans: Because it is the target to be predicted. Including it in X would
# cause target leakage.
#
# Q9. Why is categorical encoding required?
# Ans: The model requires numeric inputs, so text categories must be
# converted into numeric representations.
#
# Q10. What is one-hot encoding?
# Ans: It represents each category with a binary indicator column.
#
# Q11. Why is drop_first=True used?
# Ans: It drops one indicator category and uses it as the reference
# category, reducing redundant columns.
#
# Q12. What is X?
# Ans: X contains the independent variables or input features.
#
# Q13. What is y?
# Ans: y contains the dependent variable or target, Exited.
#
# Q14. Why do we split the dataset?
# Ans: To train the model on one portion and evaluate it on unseen data.
#
# Q15. Why is test_size=0.20 used?
# Ans: It reserves 20% of the data for final testing and uses 80% for
# training and validation.
#
# Q16. What is random_state=42?
# Ans: It fixes the random split so the same split can be reproduced.
#
# Q17. What is stratify=y?
# Ans: It keeps approximately the same class proportions in training
# and testing subsets as in the original target.
#
# Q18. What is feature scaling?
# Ans: It transforms feature values to a common scale so features with
# large numeric ranges do not dominate only because of their magnitude.
#
# Q19. Why is StandardScaler used?
# Ans: It standardizes features by subtracting their mean and dividing
# by their standard deviation.
#
# Q20. Why do we fit the scaler only on training data?
# Ans: To avoid data leakage from the test data.
#
# Q21. What is an ANN?
# Ans: An Artificial Neural Network is a model made of connected
# computational units called neurons, arranged in layers.
#
# Q22. What is a neuron?
# Ans: A neuron calculates a weighted sum of inputs, adds a bias and
# applies an activation function.
#
# Q23. What is a Dense layer?
# Ans: It is a fully connected layer in which every neuron receives
# input from all outputs of the preceding layer.
#
# Q24. How many hidden Dense layers are in this model?
# Ans: Two: the first has 64 neurons and the second has 32 neurons.
#
# Q25. What is the purpose of the 64-neuron layer?
# Ans: It learns patterns from the input features. The number 64 is a
# model-design choice.
#
# Q26. Why is ReLU used in hidden layers?
# Ans: ReLU is computationally simple and introduces non-linearity.
# It outputs max(0, x).
#
# Q27. Why is sigmoid used in the output layer?
# Ans: It produces a value between 0 and 1, suitable for estimating
# the probability of the positive class in binary classification.
#
# Q28. Why does the output layer contain only one neuron?
# Ans: A single sigmoid output is sufficient for a binary classifier.
#
# Q29. What is dropout?
# Ans: It is a regularization method that randomly disables a fraction
# of neuron outputs during training to help reduce overfitting.
#
# Q30. What does Dropout(0.30) mean?
# Ans: It randomly sets approximately 30% of the inputs to that dropout
# layer to zero during training.
#
# Q31. What is overfitting?
# Ans: It is when a model fits training data too closely and performs
# poorly on unseen data.
#
# Q32. What is underfitting?
# Ans: It is when a model has not learned the underlying patterns well
# and performs poorly even on training data.
#
# Q33. What is model.compile() used for?
# Ans: It configures the optimizer, loss function and metrics used
# during training.
#
# Q34. What is the Adam optimizer?
# Ans: It is an adaptive gradient-based optimization algorithm used
# to update neural-network weights.
#
# Q35. What is binary cross-entropy?
# Ans: It is a loss function used to measure prediction error in
# binary classification.
#
# Q36. What is an epoch?
# Ans: One complete pass through the training data.
#
# Q37. What is batch_size=32?
# Ans: The model processes 32 records per batch before updating weights.
#
# Q38. What is validation data?
# Ans: Data kept aside during training to monitor how well the model
# generalizes to records not used for weight updates.
#
# Q39. What does validation_split=0.20 mean?
# Ans: Keras reserves 20% of the data supplied to model.fit() for
# validation.
#
# Q40. What is EarlyStopping?
# Ans: A callback that stops training when the monitored metric fails
# to improve for the specified patience.
#
# Q41. What does patience=10 mean?
# Ans: Training can continue for 10 epochs without improvement in the
# monitored validation loss before stopping.
#
# Q42. What does restore_best_weights=True do?
# Ans: It restores the model weights from the epoch with the best
# monitored validation loss.
#
# Q43. What does model.predict() return here?
# Ans: A probability prediction for each test record, because the
# output layer uses sigmoid.
#
# Q44. Why is the threshold 0.5 used?
# Ans: It is a common default decision threshold: probability at least
# 0.5 is classified as 1; otherwise it is classified as 0.
#
# Q45. What does astype(int) do?
# Ans: It converts Boolean threshold results into integer class labels
# 0 and 1.
#
# Q46. What does ravel() do?
# Ans: It converts the prediction array into a one-dimensional array.
#
# Q47. What is accuracy?
# Ans: The proportion of total predictions that are correct.
#
# Q48. What is a confusion matrix?
# Ans: A table that compares actual and predicted classes and reports
# true negatives, false positives, false negatives and true positives.
#
# Q49. What is a false positive in this project?
# Ans: The model predicts that a customer will exit, but the customer
# actually does not exit.
#
# Q50. What is a false negative in this project?
# Ans: The model predicts that a customer will stay, but the customer
# actually exits.
#
# Q51. Which error could be costly for a bank?
# Ans: A false negative may be costly because the bank fails to identify
# a customer who is actually going to leave. The business cost depends
# on the bank's use case.
#
# Q52. Is accuracy always a sufficient metric?
# Ans: No. If the classes are imbalanced, a model may have high accuracy
# while performing poorly on the minority class. Precision, recall,
# F1-score and ROC-AUC can provide more information.
#
# Q53. What is recall?
# Ans: Recall = TP / (TP + FN). It measures how many actual positive
# cases were correctly identified.
#
# Q54. What is precision?
# Ans: Precision = TP / (TP + FP). It measures how many predicted
# positive cases were actually positive.
#
# Q55. What is the difference between validation and testing?
# Ans: Validation is used during model development and monitoring.
# The test set is held out for final evaluation.
#
# Q56. Does the test set take part in model.fit()?
# Ans: No. The notebook trains on X_train and y_train. X_test is used
# later for prediction and evaluation.
#
# Q57. What is backpropagation?
# Ans: It is the process of calculating gradients of the loss with
# respect to network parameters so the optimizer can update them.
#
# Q58. What is an epoch versus an iteration?
# Ans: An epoch is one pass through the training data. An iteration
# is one parameter update, usually after processing one batch.
#
# Q59. Can a neural network solve non-linear problems?
# Ans: Yes. Hidden layers with non-linear activation functions allow
# the network to learn non-linear relationships.
#
# Q60. Why might the result change on another run?
# Ans: Neural-network initialization, data shuffling and other
# stochastic training operations can cause results to vary.
#
# Q61. What is the purpose of model.summary()?
# Ans: It displays each layer, its output shape and the number of
# parameters.
#
# Q62. What is the difference between training and inference?
# Ans: Training adjusts model weights using data. Inference uses the
# trained model to make predictions without updating its weights.
#
# Q63. What happens to Dropout during prediction?
# Ans: Dropout is inactive during inference; all units are used with
# the appropriate scaling handled by the framework.
#
# Q64. What is the limitation of this notebook's evaluation?
# Ans: It reports accuracy and a confusion matrix only. It does not
# report precision, recall, F1-score, ROC-AUC or a classification report.
#
# Q65. How can this project be improved?
# Ans: Evaluate precision, recall, F1-score and ROC-AUC; inspect class
# imbalance; tune the classification threshold using validation data;
# compare with baseline models; and tune the network hyperparameters.
#
# ============================================================
# 6. SAMPLE DATA / INPUT FORMAT
# ============================================================
#
# This notebook reads records from a CSV file. It does not request
# keyboard input using input().
#
# The CSV must contain these columns because the code refers to them:
# - RowNumber
# - CustomerId
# - Surname
# - Geography
# - Gender
# - Exited
#
# The remaining columns retained in X must be numeric after the
# Geography and Gender columns are encoded. In the commonly used
# Churn_Modelling.csv dataset, the retained fields include customer
# financial/account features such as CreditScore, Age, Tenure, Balance,
# NumOfProducts, HasCrCard, IsActiveMember and EstimatedSalary.
#
# ILLUSTRATIVE CSV HEADER:
# RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
#
# ILLUSTRATIVE FIRST RECORD:
# 1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
#
# - The row above is only an example of the expected CSV-style structure.
# - Save the complete dataset as Churn_Modelling.csv.
# - Ensure that the file's column names match the code.
# - The exact dataset rows and resulting accuracy depend on the file used.
#
# ============================================================
# 7. HOW TO RUN THE NOTEBOOK
# ============================================================
#
# REQUIREMENTS:
# - Python
# - Jupyter Notebook or JupyterLab
# - pandas
# - scikit-learn
# - TensorFlow/Keras
#
# WINDOWS (COMMAND PROMPT OR POWERSHELL):
#
# 1. Open a terminal in the project folder.
#
# 2. Create a virtual environment:
#    py -m venv .venv
#
# 3. Activate it:
#    .venv\Scripts\activate
#
# 4. Upgrade pip:
#    python -m pip install --upgrade pip
#
# 5. Install dependencies:
#    python -m pip install pandas scikit-learn tensorflow jupyter
#
# 6. Start JupyterLab:
#    jupyter lab
#
# 7. Open bank-churn-neural-nw.ipynb in the browser and run the cells
#    from top to bottom.
#
# LINUX (UBUNTU):
#
# 1. Open Terminal in the project folder.
#
# 2. Create a virtual environment:
#    python3 -m venv .venv
#
# 3. Activate it:
#    source .venv/bin/activate
#
# 4. Upgrade pip:
#    python -m pip install --upgrade pip
#
# 5. Install dependencies:
#    python -m pip install pandas scikit-learn tensorflow jupyter
#
# 6. Start JupyterLab:
#    jupyter lab
#
# 7. Open bank-churn-neural-nw.ipynb and run the cells from top to bottom.
#
# DATASET PATH REMINDER:
# - The original notebook currently uses:
#   C:/Sarang/College/lp3/ml/Churn_Modelling.csv
# - That path will only work if the CSV exists at that exact path.
# - On Windows, update the path in the data-loading cell if the CSV is
#   stored elsewhere.
# - On Linux, use the correct Linux path, for example:
#   data/Churn_Modelling.csv
#   (provided the CSV is inside a data folder in the notebook directory).
# - This path change is necessary for portability; it has deliberately
#   not been made to the original code in this annotated copy.
#
# ============================================================
# 8. QUICK REVISION BEFORE THE EXTERNAL VIVA
# ============================================================
#
# Project: Bank Customer Churn Prediction
# Task: Supervised Binary Classification
# Model: Artificial Neural Network (ANN)
# Framework: Keras
# Model type: Sequential
# Hidden layers: 64 ReLU neurons, then 32 ReLU neurons
# Regularization: Dropout 0.30 and 0.20
# Output layer: 1 neuron with Sigmoid
# Optimizer: Adam
# Loss: Binary Cross-Entropy
# Metric during training: Accuracy
# Data split: 80% training portion / 20% test portion
# Validation: 20% of the training portion passed to model.fit()
# Scaling: StandardScaler
# Encoding: One-hot encoding using pandas get_dummies()
# Training callback: EarlyStopping
# Maximum epochs: 100
# Batch size: 32
# Prediction threshold: 0.5
# Evaluation: Accuracy and Confusion Matrix
#
# MOST IMPORTANT CONCEPTS TO REMEMBER:
# 1. Exited is the target variable.
# 2. X contains input features and y contains the target.
# 3. One-hot encoding converts Geography and Gender into numeric columns.
# 4. StandardScaler uses training-set statistics to scale both train and test.
# 5. ReLU is used in hidden layers.
# 6. Sigmoid is used for the binary output probability.
# 7. Binary cross-entropy is the loss function.
# 8. Adam is the optimizer.
# 9. Dropout helps reduce overfitting.
# 10. EarlyStopping monitors validation loss.
# 11. Accuracy measures the fraction of correct predictions.
# 12. Confusion matrix shows TN, FP, FN and TP.
#
# ============================================================
# END OF VIVA NOTES
# ============================================================